In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver # to save the memory in ram
from langchain_core.messages import SystemMessage,HumanMessage


In [19]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
DEEPSEEK_API_KEY = os.getenv("DEEP_SEEK_API_KEY")

In [20]:
model = ChatGoogleGenerativeAI(
    # model="gemini-2.5-flash",
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY
)

In [21]:
class ChatState (TypedDict):

    messages: Annotated[list[BaseMessage], add_messages] # To append messages into list its recommended to use add_messages from langgraph instead of operator.add

In [22]:
def chat_node(state: ChatState): 

    # take user query from state
    messages = state['messages']

    # send to llm
    response = model.invoke(messages)

    # response store to state
    return {'messages' : [response]}

In [23]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)
chatbot = graph.compile(checkpointer=checkpointer)


In [24]:
# initial_state ={
#     'messages' : [HumanMessage(content='What is the capital of pakistan')]
# }

# response = chatbot.invoke(initial_state)['messages'][-1].content
# print('response',response)

In [26]:
thread_id = '1'
while True:
    user_message = input('Type here: ')

    print('User:', user_message)

    if user_message.strip().lower() in ['exit','quit','bye']:
        break

    config= {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)

    print('AI :', response['messages'][-1].content)

User: hey 
AI : I am a large language model, trained by Google.
User: my name is faraz
AI : Yes, Faraz, you've told me your name. It's nice to remember it! Is there anything else you'd like to talk about or ask me?
User: i am 28 years old
AI : That's great to know, Faraz! Thanks for sharing. Is there anything you'd like to do with that information, or would you like to talk about something else?
User: what is my name and my age?
AI : Your name is **Faraz**, and you are **28 years old**.
User: exit
